In [1]:
import h5py
from itertools import product
import os
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm

# Load neural response data

# Load behavioral data

file_path1 = 'face10156_z_correlations_real_si.h5'

def load_data(file_path):
    """Function to load data from subarray_0 to subarray_399 from a given file path."""
    with h5py.File(file_path, 'r') as file:
        data_list = []
        for i in range(400):  # Load from subarray_0 to subarray_399
            dataset_name = f'z_corr_matrix_{i}'
            if dataset_name in file:
                data = file[dataset_name][:]
                data_list.append(data)
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
        return data_list

# Load data for each file
all_z_correlations_real = load_data(file_path1)

msub_r_list = []
for brain_area in range(400):
    z_corr_matrix_real = all_z_correlations_real[brain_area]
    corr_sametrial_stacked = []

    for matrix in z_corr_matrix_real:
        # Extract the diagonal of the submatrices
        submatrix = np.diag(matrix)
        # Append to the list for current brain area
        corr_sametrial_stacked.append(submatrix)

    # Append the list of stacked arrays to the main list
    msub_r_list.append(corr_sametrial_stacked)
    
realpair = np.array(msub_r_list)  # Convert to numpy array


import h5py
from itertools import product
import os
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

#Define file paths for saving
file_path6 = 'face10156_cross_correlations_si.h5'

def load_data1(file_path):
    """Function to load data from datasets with names general_s_i and general_r_i   
    (where i ranges from 0 to 399) from a given file path, and store them in separate lists."""
    
    data1_list = []  # List to store data from general_s_ datasets
    data2_list = []  # List to store data from general_r_ datasets
    data3_list = []  # List to store data from general_s_ datasets
    data4_list = []  # List to store data from general_r_ datasets

    with h5py.File(file_path, 'r') as file:
        for i in range(400):  # Iterate over indices from 0 to 399
            s1_dataset_name = f'all_correlations_sarb_{i}'
            r1_dataset_name = f'all_correlations_rbsa_{i}'
            s2_dataset_name = f'all_correlations_sbra_{i}'
            r2_dataset_name = f'all_correlations_rasb_{i}'

            # Check and read general_s_ dataset
            if s1_dataset_name in file:
                data1 = file[s1_dataset_name][:]
                data1_list.append(data1)
            else:
                print(f"Dataset '{s1_dataset_name}' not found in the file.")

            # Check and read general_r_ dataset
            if r1_dataset_name in file:
                data2 = file[r1_dataset_name][:]
                data2_list.append(data2)
            else:
                print(f"Dataset '{r1_dataset_name}' not found in the file.")

            # Check and read general_s_ dataset
            if s2_dataset_name in file:
                data3 = file[s2_dataset_name][:]
                data3_list.append(data3)
            else:
                print(f"Dataset '{s2_dataset_name}' not found in the file.")

            # Check and read general_r_ dataset
            if r2_dataset_name in file:
                data4 = file[r2_dataset_name][:]
                data4_list.append(data4)
            else:
                print(f"Dataset '{r2_dataset_name}' not found in the file.")
        return data1_list, data2_list, data3_list, data4_list

# Load data for each file
s1,r1,s2,r2 = load_data1(file_path6)

crosspairs=[]

for brain_area in range(400):
    # For each brain region, extract the corresponding data
    s1data = s1[brain_area]  # shape (23, 24, number_of_voxels)
    r1data = r1[brain_area]  # shape (23, 24, number_of_voxels)
    s2data = s2[brain_area]  # shape (23, 24, number_of_voxels)
    r2data = r2[brain_area]  # shape (23, 24, number_of_voxels)

    crosspaira=[]
    crosspairb=[]

    for sub in range(23):
        sub_s1data=np.mean(s1data[sub],axis=0)
        sub_r1data=np.mean(r1data[sub],axis=0)
        sub_s2data=np.mean(s2data[sub],axis=0)
        sub_r2data=np.mean(r2data[sub],axis=0)
        sa=(sub_s1data+sub_r1data)/2
        sb=(sub_s2data+sub_r2data)/2
        crosspaira.append(np.diag(sa))
        crosspairb.append(np.diag(sb))

    crosspair=np.vstack((crosspaira,crosspairb))
    crosspairs.append(crosspair)

In [ ]:
crosspairs=np.array(crosspairs)
realpair=np.array(realpair)

In [ ]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import multiprocessing as mp
from functools import partial
import os

# Enable Pandas and R DataFrame interoperability
pandas2ri.activate()

# Define function to process a single brain region
def process_region(region, realpair, crosspairs):
    import pandas as pd
    import numpy as np
    import os
    from rpy2.robjects import pandas2ri
    import rpy2.robjects as ro
    pandas2ri.activate()

    process_id = os.getpid()
    print(f"Process {process_id} - Processing region {region}")
    
    try:
        # === 1. Construct paired data, each row represents a trial-pair ===
        data_list = []
        for subject in range(46):
            for t in range(24):
                condition = 1 if t < 12 else 2
                stimulus_group = 2 if subject >= 23 else 1
                y_real = realpair[region, subject, t]
                y_pseudo = crosspairs[region, subject, t]
                data_list.append([subject, t, condition, y_real, y_pseudo, stimulus_group])
        
        df_pairs = pd.DataFrame(data_list, columns=["Subject", "Trial", "Condition", "y_real", "y_pseudo", "StimulusGroup"])
        
        df_pairs["z_real"] = df_pairs.groupby("Subject")["y_real"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
        df_pairs["z_pseudo"] = df_pairs.groupby("Subject")["y_pseudo"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
        
        # === 3. Remove pairs where either side is an outlier ===
        df_pairs["is_outlier"] = (df_pairs["z_real"].abs() > 3) | (df_pairs["z_pseudo"].abs() > 3)
        df_pairs_clean = df_pairs[~df_pairs["is_outlier"]].copy()
        
        if df_pairs_clean.empty:
            print(f"Process {process_id} - Skipping region {region}: all data removed after cleaning")
            return None
        
        # === 4. Reshape into the format required by lmer ===
        df_long = pd.concat([
            df_pairs_clean.assign(PairType=1, y=df_pairs_clean["y_real"]),
            df_pairs_clean.assign(PairType=0, y=df_pairs_clean["y_pseudo"])
        ], ignore_index=True)
        
        df_long = df_long[["Subject", "Trial", "Condition", "StimulusGroup", "PairType", "y"]]
        df_long["Subject"] = df_long["Subject"].astype(str)
        df_long["Trial"] = df_long["Trial"].astype(str)

        df_long["y"] = df_long["y"] * 10  # Scale up to avoid numerical precision issues

        # === 5. Convert to R dataframe ===
        r_df = pandas2ri.py2rpy(df_long)
        ro.globalenv["df"] = r_df

        # === 6. R code: fit lmer + permutation ===
        r_code = """
        library(lme4)
        df$PairType <- as.factor(df$PairType)
        df$Condition <- as.factor(df$Condition)
        df$StimulusGroup <- as.factor(df$StimulusGroup)
        df$Subject <- as.factor(df$Subject)
        df$Trial <- as.factor(df$Trial)

        model_full <- lmer(y ~ Condition + PairType + StimulusGroup + (1 | Subject) + (1 | Trial), data=df, REML=FALSE)

        singular <- isSingular(model_full)
        if (singular) warning("Singular fit detected")

        real_pairtype_coef <- fixef(model_full)["PairType1"]
        N_PERMUTATIONS <- 3000
        perm_pairtype_coefs <- numeric(N_PERMUTATIONS)
        fail_count <- 0

        for (i in 1:N_PERMUTATIONS) {
            perm_df <- df
            for (subj in unique(df$Subject)) {
                for (cond in unique(df$Condition)) {
                    for (grp in unique(df$StimulusGroup)) {
                        idx <- which(df$Subject == subj & df$Condition == cond & df$StimulusGroup == grp)
                        if (length(idx) > 1) {
                            perm_df$PairType[idx] <- sample(df$PairType[idx])
                        }
                    }
                }
            }
            perm_df$PairType <- factor(perm_df$PairType, levels=c("0", "1"))
            perm_model <- tryCatch(
                lmer(y ~ Condition + PairType + StimulusGroup + (1 | Subject) + (1 | Trial), data=perm_df, REML=FALSE),
                error = function(e) { fail_count <<- fail_count + 1; return(NA) }
            )
            if (is.na(perm_model)[1]) next
            perm_pairtype_coefs[i] <- fixef(perm_model)["PairType1"]
        }

        perm_pairtype_coefs <- perm_pairtype_coefs[!is.na(perm_pairtype_coefs)]
        p_value_pairtype_perm <- mean(perm_pairtype_coefs >= real_pairtype_coef)

        list(
            intercept=fixef(model_full)["(Intercept)"],
            pair=fixef(model_full)["PairType1"],
            condition2=fixef(model_full)["Condition2"],
            stimgroup2=fixef(model_full)["StimulusGroup2"],
            p_value_pairtype_perm=p_value_pairtype_perm,
            singular=singular,
            fail_count=fail_count
        )
        """
        
        r_results = ro.r(r_code)
        results = [region] + list(r_results)
        print(f"Process {process_id} - Completed region {region}")
        return results

    except Exception as e:
        print(f"Process {process_id} - Error processing region {region}: {e}")
        return None


# Main function - parallel processing
def run_analysis(realpair, crosspairs):
    # Set number of cores for parallel processing
    num_cores = 10
    print(f"Running with {num_cores} cores")
    
    # Create process pool
    pool = mp.Pool(processes=num_cores)
    
    # Create partial function with fixed realpair and crosspairs parameters
    process_func = partial(process_region, realpair=realpair, crosspairs=crosspairs)
    
    # Process all brain regions in parallel
    results = pool.map(process_func, range(400))
    
    # Close process pool
    pool.close()
    pool.join()
    
    # Filter out None values
    results_list = [r for r in results if r is not None]
    
    # Create results DataFrame
    df_results = pd.DataFrame(results_list, columns=[
        "Brain_Region", "Intercept", "Pair_Effect", 
        "Condition_Effect", "StimulusGroup_Effect",
        "PairType_p_perm", "Singular_Fit", "Fail_Count"
    ])
    
    # Save results
    #df_results.to_csv("real_vs_pseudo_results_global_perm_si.csv", index=False) 
    
    return df_results

# Usage:
results = run_analysis(np.array(realpair), np.array(crosspairs))

In [ ]:
from statsmodels.stats.multitest import multipletests

# Perform FDR correction (correct PairType_p across all brain regions)
results["PairType_p_perm_FDR"] = multipletests(results["PairType_p_perm"], method="fdr_bh")[1]
significant_regions1 = results[
    (results["PairType_p_perm_FDR"] < 0.05)]

print(significant_regions1)
# Extract significant brain region numbers
significant_region_numbers = significant_regions1["Brain_Region"].tolist()

# Print brain region numbers
print(significant_region_numbers)

In [ ]:
# Save as CSV file
results.to_csv("sireal_0526.csv", index=False)

print("Results saved to 'sireal_0526.csv'")